# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
# Load the data
df = pd.read_csv("AviationData.csv", encoding = "latin1")

# check for data types
df.info()

# Summary Statistics
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

C:\Users\HomePC\AppData\Local\Temp\ipykernel_16124\173043250.py:2: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("AviationData.csv", encoding = "latin1")


,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
# Inspeting all colomns to idenfify relevant ones
df.columns


Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='object')

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
# To calculate total number of passengers, add the total from injury columns
injury_cols = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

# Already inspected all the injury columns data type is float
# Fill any missing value with 0 - assumption, if missing then injury is 0
df[injury_cols] = df[injury_cols].fillna(0)

#Derive total number of passengers
df["Total.Passengers"] = (df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"] + df["Total.Minor.Injuries"] + df["Total.Uninjured"])
df = df[df["Total.Passengers"]>0] # Incase some still and zero

# serious/fatal injury rate - (serious+fatal)/total passengers
df["Serious+Fatal"] = df["Total.Serious.Injuries"] + df["Total.Fatal.Injuries"]
df["Serious.Fatal.Injury.Rate"] = df["Serious+Fatal"]/df["Total.Passengers"]
df["Serious.Fatal.Injury.Rate"].describe()

count    87580.000000
mean         0.290022
std          0.437306
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000
Name: Serious.Fatal.Injury.Rate, dtype: float64

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:
# Since the client is only interested with  professional airplane;
df = df[df["Aircraft.Category"] == "Airplane"] # removes non-airplane
df =df[df["Amateur.Built"] == "No"] # removes amateur builds

# Assuming max liteme of 40 years; >1983
df = df.dropna(subset=["Event.Date"]) # drop any null values
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors = "coerce")
df["Year"] = df["Event.Date"].dt.year #Derive column for year to filter > 1983
df = df[df["Year"] >= 1983]

# Check for the values in the aircraft damage column
df["Aircraft.damage"].value_counts(dropna=False)

# Drop missing values because we only need ones with damage information
df = df.dropna(subset=["Aircraft.damage"])

# Derive new column to track whether airplane was detroyed or not
df["Aircraft.damage"] = df["Aircraft.damage"].str.strip().str.title() # to standardize str format
df["Destroyed"] = (df["Aircraft.damage"] == "Destroyed").astype(int) # If damaged 1, if not 0
df["Destroyed"].value_counts()

Destroyed
0    17488
1     2268
Name: count, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [6]:
# Drop nulls
df = df.dropna(subset=["Make"])

# Standardize text
df["Make"] = df["Make"].str.strip().str.upper()
df["Make"].value_counts().head(20) # to inspect


# keep Makes with >= 50
make_counts = df["Make"].value_counts()
valid_make = make_counts[make_counts >= 50].index
df = df[df["Make"].isin(valid_make)]
df["Make"].value_counts() # Inspect


Make
CESSNA                            6976
PIPER                             3930
BEECH                             1395
BOEING                             486
MOONEY                             357
AIR TRACTOR INC                    219
BELLANCA                           218
CIRRUS DESIGN CORP                 217
MAULE                              215
AIR TRACTOR                        203
AERONCA                            200
CHAMPION                           157
GRUMMAN                            145
LUSCOMBE                           139
STINSON                            129
CIRRUS                             128
NORTH AMERICAN                     104
TAYLORCRAFT                         93
DEHAVILLAND                         92
AERO COMMANDER                      88
EMBRAER                             86
AVIAT AIRCRAFT INC                  75
DIAMOND AIRCRAFT IND INC            74
SOCATA                              72
AVIAT                               70
RAYTHEON AIRCRAFT CO

### Cleaning tasks Explained
1. Drop Nulls - removed all rows with missing values for Make acolumn
2. Standardized the column to by removing any trailing/leading spaces and convert to uppercase, for consistency

    In between, I inspected the columns and to ensure I am on the right track

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [7]:
# Drop nulls
df = df.dropna(subset=["Model"])

# Standardize text
df["Model"] = df["Model"].str.strip().str.upper()
df["Model"].value_counts().head(20) # inspect

# check unique and how models and makes relate
df[["Model","Make"]].nunique()
df.groupby("Make")["Model"].nunique().sort_values(ascending=False).head(20)

# Derive new column "make_model"
df["Make_Model"] = df["Make"] + "_" + df["Model"]
df["Make_Model"].value_counts()



Make_Model
CESSNA_172            745
CESSNA_152            308
CESSNA_182            299
CESSNA_172S           267
PIPER_PA28            263
                     ... 
AERO COMMANDER_685      1
EMBRAER_EMB-190         1
MAULE_MX7-180A          1
BOEING_777-200ER        1
PIPER_PA-44             1
Name: count, Length: 1898, dtype: int64

1. Derived a new column that takes both Make and Model (the unique identifier for a given plane type), which can be later helpful when conducting analysis
2. Filter aircrafts  using the new column "Make_Model" with those that have recorded at least 50 accidents. Bigger sample is better for analysis

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [8]:
# Keeping columns as they are but making sure to standardize necessary ones
# first inspect the columns using value_counts
df["Engine.Type"].value_counts(dropna=False)
df["Engine.Type"] = df["Engine.Type"].str.strip().str.title()

df["Weather.Condition"].value_counts(dropna=False)
df["Weather.Condition"] = df["Weather.Condition"].str.strip().str.upper() # upper since coded VMC,IMC,UNK

df["Number.of.Engines"].value_counts(dropna=False)
df["Number.of.Engines"] = pd.to_numeric(df["Number.of.Engines"], errors="coerce") # ensure its of numeric type

df["Purpose.of.flight"].value_counts(dropna=False)
df["Purpose.of.flight"] = df["Purpose.of.flight"].str.strip().str.title() # for consistency

df["Broad.phase.of.flight"].value_counts(dropna=False)
df["Broad.phase.of.flight"] = df["Broad.phase.of.flight"].str.strip().str.title()

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [9]:
# to check count of null values in each column for df
nan_counts = df.isna().sum().sort_values(ascending=False)
nan_counts
## The important columns cleaned above have 0 NaNs. Schedule has the most at 15083

# Inspect percentage
nan_percent = (df.isna().mean() * 100).sort_values(ascending=False)
nan_percent

# After inspection, Schedule (92%), Broad.phase.of.flight (85%), and air.carrier (54%) are the only ones above 50%
# The rest could potentially be used to explain the findings after anlaysis
# Desicion made - drop columns >50% with NaNs
cols_to_drop = nan_percent[nan_percent > 50].index

df = df.drop(columns=cols_to_drop)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16370 entries, 4150 to 88886
Data columns (total 34 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Event.Id                   16370 non-null  object        
 1   Investigation.Type         16370 non-null  object        
 2   Accident.Number            16370 non-null  object        
 3   Event.Date                 16370 non-null  datetime64[ns]
 4   Location                   16367 non-null  object        
 5   Country                    16369 non-null  object        
 6   Latitude                   15360 non-null  object        
 7   Longitude                  15356 non-null  object        
 8   Airport.Code               11154 non-null  object        
 9   Airport.Name               11252 non-null  object        
 10  Injury.Severity            16370 non-null  object        
 11  Aircraft.damage            16370 non-null  object        
 12  Aircra

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [10]:
cleaned_aviation_data = df.to_csv("data/cleaned_aviation_data.csv", index=False)